In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os
from tqdm import tqdm
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split

%matplotlib inline

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
csv_path = os.path.join(path, "Q1_data.csv")

df = pd.read_csv(csv_path)

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
# 1. What does our target variable (charges) look like?
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df = df.drop(columns="Order_ID", axis=1)

In [ ]:
# Task 2: Write your code here:
df.isnull().sum()

In [ ]:
df = df.dropna(subset=['Delivery_Time'])

In [ ]:
df.isnull().sum()

In [ ]:
df["Time_of_Day"] = df["Time_of_Day"].fillna(df["Time_of_Day"].mode()[0])
df["Weather"] = df["Weather"].fillna(df["Weather"].mode()[0])
df["Traffic_Level"] = df["Traffic_Level"].fillna(df["Traffic_Level"].mode()[0])
df["Courier_Experience_yrs"] = df["Courier_Experience_yrs"].fillna(df["Courier_Experience_yrs"].mode()[0])

In [ ]:
df.isnull().sum()

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df)

In [ ]:
# Task 4: Write your code here: Encode categorical variables if needed (Bonus if used One Hot Encoding)
df_OH = df.copy()
toOH = ["Vehicle_Type", "Time_of_Day", "Traffic_Level", "Weather"]
onehot_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

In [ ]:
df_OH = pd.DataFrame(onehot_encoder.fit_transform(df_OH[toOH]), columns=onehot_encoder.get_feature_names_out(df_OH[toOH].columns))

In [ ]:
df_OH.info()

In [ ]:
# Task 5: Write your code here:

In [ ]:
X

In [ ]:
X = pd.DataFrame(df_OH)
y = df[["Delivery_Time"]]

In [ ]:
scaler = StandardScaler()
X = scaler.fit_transform(X)

scaler = StandardScaler()
y = scaler.fit_transform(y)

In [ ]:
#task 6
#y.hist()

In [ ]:
# Task 1: Write your code here:
#be for scaling in Task 5

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error as mean_absolute_error

In [ ]:
from sklearn.ensemble import RandomForestRegressor


In [ ]:
models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
}

In [ ]:
all_results = {}

for name in models:
  all_results[name] = {'mae': []}

In [ ]:
# Task 2,3,4,5: Write your code here:
n_splits=5
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)


    # Store results
    all_results[model_name]["mae"].append(mae)

In [ ]:
# Task 1: Write your code here:
coeffs = {}

coeffs['RFR'] = models['Random Forest Regressor'].feature_importances_

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, coef) in enumerate(coeffs.items()):
  # Sort features by absolute coefficient value
  absolute_coef = np.abs(coef)
  sorted_idx = np.argsort(absolute_coef)

  ax = axes[i]
  ax.barh(features[sorted_idx], coef[sorted_idx])
  ax.set_title(f"{model_name} Coefficients")
  ax.set_xlabel("Coefficient Value (Impact)")

plt.tight_layout()
plt.show()

In [ ]:
# Task 2: Write your code here:
y_pred = model.predict(X_test)
plt.hist(y_pred)

In [ ]:
%pip install kagglehub catboost lightgbm tqdm -q


In [ ]:
from catboost import CatBoostRegressor

In [ ]:
models = {
  "Random Forest Regressor": RandomForestRegressor(n_estimators=200),
  "CatBoost": CatBoostRegressor(verbose=0)
}

In [ ]:
all_results = {}
for name in models:
  all_results[name] = {'mae': []}

In [ ]:
# Task 2,3,4,5: Write your code here:

n_splits=5
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  for model_name, model in models.items():
    print(f"Training {model_name}...")

    # Train
    model.fit(X_train, y_train)

    # Predict
    y_pred = model.predict(X_test)

    # Calculate metrics
    mae = mean_absolute_error(y_test, y_pred)


    # Store results
    all_results[model_name]["mae"].append(mae)